# 04 · Where heat vulnerability overlaps

**Objective.** Compare heat vulnerability with three quantities across New York City's 178 modified ZIP code
areas (MODZCTA): applications to the Cooling Assistance program in 2025, the share of residents who are Black
and non-Hispanic, and the share who are Hispanic of any race. Each comparison is a bivariate map: color
encodes the heat-vulnerability tercile and the comparison variable's tercile at once.

| Inputs (`../inputs/`) | Source |
|---|---|
| `modzcta.geojson` | NYC Open Data pri4-ifjk, MODZCTA polygons with `pop_est` |
| `tracts2020.geojson` | NYC Open Data 63ge-mke6, 2020 census tracts (carry their NTA code) |
| `hvi_nta2020.csv` | NYC DOHMH Heat Vulnerability Index by 2020 NTA |
| `dcp_decennial_census_2020.xlsx` | NYC DCP Decennial Census workbook (CT2020, NTA2020 and NYC2020 rows) |
| `dss_applications_2025_by_modzcta.csv` | DSS applications aggregated by `aggregate_dss.ipynb` |

Outputs go to `data/`: `hvi_by_modzcta.csv`, `modzcta_bivariate.geojson`, `map_values.csv`, `legend.json`,
`citywide_context.json`.

**The geography problem.** The HVI is published by neighborhood (NTA) and the applications by ZIP code, and
the two do not nest. Both are brought to MODZCTA, the health department's standard geography for ZIP-keyed
data: applications by summing ZIP counts upward (`aggregate_dss.ipynb`), the HVI and the Census counts by areal
interpolation of census tracts (steps 1 and 2 below).

In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()                      # run this notebook from its own directory (run_notebooks.py does)
ROOT = HERE.parent
sys.path.insert(0, str(ROOT))
DATA = HERE / "data"
DATA.mkdir(exist_ok=True)

import geopandas as gpd
import numpy as np
import pandas as pd

from common import BIVARIATE_PALETTE, INPUTS, NO_DATA, SOURCES, load_hvi, write_geojson, write_json

NYC_POPULATION_2020 = 8804190  # Source: DCP 2020 Census workbook, NYC2020 row, Pop1
MAP_SOURCES = [SOURCES["dss"], SOURCES["crosswalk"], SOURCES["hvi"], SOURCES["census"], SOURCES["modzcta"], SOURCES["tracts"]]


## 1. Heat vulnerability at MODZCTA level

2020 census tracts nest exactly in 2020 NTAs (the tract file carries the `nta2020` code), so every tract
inherits its NTA's HVI rank. Tracts do not nest in MODZCTA, so each tract is split among the areas it
overlaps by **areal interpolation**: the two layers are overlaid in a projected coordinate system
(EPSG:2263, New York–Long Island State Plane), and each piece of a tract carries the fraction of the
tract's area that it covers. A tract that straddles a boundary therefore contributes to both sides in
proportion to area, rather than being assigned whole to the side containing one point. This assumes
residents are spread uniformly within a tract, which is the weakest assumption available at tract
resolution.

Two refinements to the plain overlay:

- MODZCTA `99999` is the dataset's pseudo-area for parks, cemeteries and airports, with a population
  estimate of zero. It is excluded as a target, so a tract's parkland does not receive a share of its
  residents.
- Tract land that lies inside no residential MODZCTA (the excluded parkland plus 4.3 square miles of
  waterfront that the MODZCTA layer does not cover) receives no residents; each tract's weights are
  renormalised to sum to one over its covered land. A tract whose overlap with residential areas is below
  1% of its area touches them only through digitising slivers (Central Park, Crotona Park and other park
  tracts, by fractions of an acre) and is treated as outside every area; these tracts drop out and their
  population is recorded in `crosswalk_stats`.

An area's `hvi_popwtd` is the mean of the ranks of its tract pieces weighted by each piece's share of 2020
tract population, over pieces with a rank and non-zero population. `nta_names` lists the neighborhoods
contributing more than 10% of an area's scored population; it is a label for readers, not an equivalence
between the two geographies. `crosswalk_stats` records the tracts left out (unscored NTAs, no overlap with
any residential area, or a rank but zero population) and how much population the split moves relative to
whole-tract assignment.

The result is a derived, descriptive mean of an ordinal index. It is not a published ZIP-level HVI and is
labeled as population-weighted wherever it is shown.


In [2]:
hvi = load_hvi()   # one row per NTA; checks the duplicated BX0802 rows agree before dropping one
tracts = gpd.read_file(INPUTS / "tracts2020.geojson")
modzcta = gpd.read_file(INPUTS / "modzcta.geojson").to_crs(4326)
census = pd.read_excel(INPUTS / "dcp_decennial_census_2020.xlsx", sheet_name="2020")
tract_pop = census.loc[census.GeoType == "CT2020", ["GeoID", "Pop1", "BNH", "Hsp1"]]
tract_pop["GeoID"] = tract_pop["GeoID"].astype(str)

t = tracts[["geoid", "nta2020", "ntaname", "geometry"]].merge(tract_pop, left_on="geoid", right_on="GeoID", how="left")
assert t.geoid.is_unique and t.Pop1.notna().all() and int(t.Pop1.sum()) == NYC_POPULATION_2020

# Areal interpolation: overlay tracts with the residential MODZCTA polygons in a projected CRS.
PSEUDO_AREA = "99999"   # parks, cemeteries and airports; pop_est 0 in the MODZCTA dataset
PROJECTED = 2263        # EPSG:2263, NY–Long Island State Plane (US feet): areas are measured here, never in degrees
targets = modzcta.loc[modzcta.modzcta.astype(str) != PSEUDO_AREA, ["modzcta", "geometry"]].to_crs(PROJECTED)
source = t.to_crs(PROJECTED)
source["tract_area"] = source.geometry.area
pieces = gpd.overlay(source, targets, how="intersection", keep_geom_type=True)   # one row per tract × area piece
pieces["share"] = pieces.geometry.area / pieces["tract_area"]                    # fraction of the tract inside this area
covered = pieces.groupby("geoid")["share"].sum()                                 # fraction of the tract inside any residential area
assert covered.max() <= 1 + 1e-4, "MODZCTA polygons overlap materially"     # they overlap by 989 sq ft citywide (digitising slivers); renormalisation absorbs it
MIN_COVERAGE = 0.01     # a tract touching residential areas by less than 1% of its area (park tracts, fractions of an acre) is outside them
covered = covered[covered >= MIN_COVERAGE]
pieces = pieces[pieces.geoid.isin(covered.index)]
pieces["weight"] = pieces["share"] / pieces["geoid"].map(covered)               # renormalised to the tract's covered land
for col in ["Pop1", "BNH", "Hsp1"]:
    pieces[col] = pieces[col] * pieces["weight"]                                 # fractional residents per piece
pieces = pieces.merge(hvi[["NTACode", "HVI_RANK"]], left_on="nta2020", right_on="NTACode", how="left")

t["covered"] = t.geoid.map(covered).fillna(0.0)
t["largest_share"] = t.geoid.map(pieces.groupby("geoid")["weight"].max()).fillna(0.0)
t = t.merge(hvi[["NTACode", "HVI_RANK"]], left_on="nta2020", right_on="NTACode", how="left")
assert abs(pieces.groupby("geoid").weight.sum().sub(1).abs().max()) < 1e-9, "weights must sum to one per tract"
scored = pieces[pieces.HVI_RANK.notna() & (pieces.Pop1 > 0)]


def weighted(group: pd.DataFrame) -> pd.Series:
    w = group.Pop1
    by_nta = group.groupby("ntaname").Pop1.sum()
    return pd.Series({"hvi_popwtd": (group.HVI_RANK * w).sum() / w.sum(),
                      "nta_names": ", ".join(sorted(by_nta[by_nta > 0.10 * w.sum()].index)),
                      "pop_scored": round(float(w.sum()), 1)})


hvi_by_modzcta = scored.groupby("modzcta").apply(weighted, include_groups=False).reset_index()
outside = t.covered == 0
crosswalk_stats = {"tracts_total": len(t), "tracts_without_hvi": int(t.HVI_RANK.isna().sum()),
                   "tracts_outside_every_modzcta": int(outside.sum()),
                   "tracts_outside_every_modzcta_with_hvi": int((outside & t.HVI_RANK.notna()).sum()),
                   "tracts_with_hvi_but_zero_population": int((t.HVI_RANK.notna() & (t.Pop1 == 0) & ~outside).sum()),
                   "tracts_scored": int((t.HVI_RANK.notna() & (t.Pop1 > 0) & ~outside).sum()),
                   "population_scored": int(round(scored.Pop1.sum())), "population_total": int(t.Pop1.sum()),
                   "population_outside_every_modzcta": int(t.loc[outside, "Pop1"].sum()),
                   "tracts_split_between_areas": int(((t.largest_share < 0.999) & ~outside).sum()),
                   "population_placed_outside_largest_area": int(round((t.Pop1 * (1 - t.largest_share))[~outside].sum())),
                   "tracts_partly_outside_all_areas": int(((t.covered < 0.999) & ~outside).sum()),
                   "population_renormalised_to_covered_land": int(round((t.Pop1 * (1 - t.covered))[~outside].sum()))}
assert crosswalk_stats["tracts_total"] == crosswalk_stats["tracts_scored"] + crosswalk_stats["tracts_without_hvi"] + \
       crosswalk_stats["tracts_outside_every_modzcta_with_hvi"] + crosswalk_stats["tracts_with_hvi_but_zero_population"], crosswalk_stats
assert abs(pieces.Pop1.sum() + crosswalk_stats["population_outside_every_modzcta"] - NYC_POPULATION_2020) < 1e-3, "population not conserved"
print(f"coverage: {crosswalk_stats['tracts_scored']} of {crosswalk_stats['tracts_total']} tracts, "
      f"{crosswalk_stats['population_scored'] / crosswalk_stats['population_total']:.2%} of residents; "
      f"{len(pieces)} tract pieces across {pieces.modzcta.nunique()} areas")
display(pd.Series(crosswalk_stats).to_frame("value"))
print("populated tracts outside every residential area (dropped):")
t.loc[outside & (t.Pop1 > 0), ["geoid", "ntaname", "Pop1"]].assign(Pop1=lambda d: d.Pop1.astype(int)).reset_index(drop=True)


coverage: 2240 of 2325 tracts, 99.91% of residents; 4262 tract pieces across 177 areas


,value
tracts_total,2325
tracts_without_hvi,71
tracts_outside_every_modzcta,11
tracts_outside_every_modzcta_with_hvi,1
tracts_with_hvi_but_zero_population,13
tracts_scored,2240
population_scored,8796527
population_total,8804190
population_outside_every_modzcta,173
tracts_split_between_areas,720


populated tracts outside every residential area (dropped):


,geoid,ntaname,Pop1
0,36085015400,Great Kills Park,5
1,36005016300,Crotona Park,34
2,36061014300,Central Park,129
3,36061000500,The Battery-Governors Island-Ellis Island-Libe...,5


## 2. Census composition at MODZCTA level

Tract counts (`Pop1`, `BNH`, `Hsp1`) are summed within each MODZCTA over the tract pieces from step 1, each
piece carrying its area share of the tract's counts, then expressed as percentages of total population,
rounded to one decimal. Summing counts first avoids averaging percentages. Because the pieces are area
shares, the interpolated population of all areas together equals the city's tract population less the
residents of tracts that overlap no residential area; `pop_2020` records each area's interpolated total.


In [3]:
sums = pieces.groupby("modzcta")[["Pop1", "BNH", "Hsp1"]].sum()
assert abs(sums.Pop1.sum() + crosswalk_stats["population_outside_every_modzcta"] - NYC_POPULATION_2020) < 1e-3
shares = pd.DataFrame({"pop_2020": sums.Pop1.round(1),
                       "pct_black_nh": (sums.BNH / sums.Pop1 * 100).round(1),
                       "pct_hispanic": (sums.Hsp1 / sums.Pop1 * 100).round(1)}).reset_index()
hvi_by_modzcta = hvi_by_modzcta.merge(shares, on="modzcta", how="left")
hvi_by_modzcta.to_csv(DATA / "hvi_by_modzcta.csv", index=False)
hvi_by_modzcta.describe().round(2)


,hvi_popwtd,pop_scored,pop_2020,pct_black_nh,pct_hispanic
count,177.00,177.00,177.00,177.00,177.00
mean,2.83,49697.89,49740.21,18.63,25.86
std,1.37,27485.12,27472.91,21.35,18.62
min,1.00,3962.90,3962.90,0.10,3.30
25%,1.63,28619.10,28619.10,3.00,11.50
50%,2.77,45349.90,45849.50,8.20,18.60
75%,4.00,68731.60,68731.60,26.80,34.60
max,5.00,116892.00,116916.00,84.10,75.90


## 3. Applications per 1,000 residents

Application counts per MODZCTA come from `aggregate_dss.ipynb`: of the 26,621 applications in 2025 that carry a
five-digit ZIP label (26,622 in the sheets' grand totals), 26,606 map to 173 areas. The rate uses the MODZCTA dataset's own population estimate, `pop_est`, so numerator and
denominator share a geography. Areas without an application record stay null: missing is not zero. The
denominator is all residents, not eligible households, so the rate does not by itself measure unmet need.

In [4]:
apps = pd.read_csv(INPUTS / "dss_applications_2025_by_modzcta.csv", dtype={"modzcta": str})
master = modzcta[["modzcta", "label", "pop_est", "geometry"]].copy()
master["modzcta"] = master["modzcta"].astype(str)
master["pop_est"] = pd.to_numeric(master.pop_est)
master = master.merge(apps, on="modzcta", how="left").merge(hvi_by_modzcta, on="modzcta", how="left")
assert len(master) == 178 and master.modzcta.is_unique
assert apps.modzcta.isin(master.modzcta).all(), "application rows must match map areas"
master["apps_1k"] = np.where(master.pop_est > 0, master.applications_2025 / master.pop_est * 1000, np.nan)
master[["applications_2025", "pop_est", "apps_1k"]].describe().round(2)


,applications_2025,pop_est,apps_1k
count,173.00,178.00,173.00
mean,153.79,47436.16,2.77
std,159.39,26810.30,2.01
min,1.00,0.00,0.11
25%,40.00,27291.50,1.33
50%,87.00,42548.00,2.07
75%,209.00,67080.50,4.00
max,674.00,112425.00,8.91


## 4. Terciles and bivariate colors

Each variable is cut at its own 1/3 and 2/3 quantiles over its non-null areas (NumPy linear interpolation),
with right-inclusive bins that include the minimum. The application class is reversed after binning so that
*fewer* applications sit to the right of the legend, matching the direction of the two demographic
comparisons (higher share to the right). Color is `palette[hvi_tercile][comparison_tercile]`; an area with
a null on either axis is gray. The exact cut points ship in `legend.json`.

In [5]:
def tercile(series: pd.Series):
    edges = np.quantile(series.dropna(), [0, 1 / 3, 2 / 3, 1])
    return pd.cut(series, edges, labels=[0, 1, 2], include_lowest=True).astype(float), edges.tolist()


master["hvi_tercile"], hvi_edges = tercile(master.hvi_popwtd)
comparisons = []
for slug, field, title, reverse, axis in [
    ("apps", "apps_1k", "Applications per 1,000 residents (2025)", True, "Fewer applications per 1,000 →"),
    ("black", "pct_black_nh", "% Black, non-Hispanic (2020 Census)", False, "Higher % Black, non-Hispanic →"),
    ("hispanic", "pct_hispanic", "% Hispanic, any race (2020 Census)", False, "Higher % Hispanic, any race →"),
]:
    ranks, edges = tercile(master[field])
    master[f"{slug}_tercile"] = 2 - ranks if reverse else ranks
    master[f"{slug}_color"] = [BIVARIATE_PALETTE[int(h)][int(v)] if pd.notna(h) and pd.notna(v) else NO_DATA
                               for h, v in zip(master.hvi_tercile, master[f"{slug}_tercile"])]
    valid = master.hvi_tercile.notna() & master[f"{slug}_tercile"].notna()
    comparisons.append({"id": slug, "field": field, "title": title, "color_column": f"{slug}_color", "class_column": f"{slug}_tercile",
                        "edges": edges, "axis_right": axis, "inverted": reverse, "classed": int(valid.sum()), "no_data": int((~valid).sum()),
                        "top_right_count": int(((master.hvi_tercile == 2) & (master[f"{slug}_tercile"] == 2)).sum())})
assert [c["classed"] for c in comparisons] == [173, 177, 177]

print("HVI tercile edges:", [round(e, 3) for e in hvi_edges])
pd.DataFrame([{"comparison": c["id"], "edges": [round(e, 3) for e in c["edges"]], "classed": c["classed"], "no data": c["no_data"], "high HVI × right column": c["top_right_count"]} for c in comparisons])


HVI tercile edges: [1.0, 2.004, 3.599, 5.0]


,comparison,edges,classed,no data,high HVI × right column
0,apps,"[0.107, 1.607, 2.98, 8.906]",173,5,3
1,black,"[0.1, 4.4, 21.6, 84.1]",177,1,50
2,hispanic,"[3.3, 13.8, 26.6, 75.9]",177,1,31


## 5. Citywide benchmarks and outputs

Benchmarks are aggregates of counts, never averages of area percentages: population shares from the Census
NYC total row; the HVI benchmark is Σ(neighborhood rank × neighborhood population) / population of scored
neighborhoods; the application benchmark is Σ applications / Σ `pop_est` × 1000 over areas with records.

In [6]:
keep = ["modzcta", "label", "nta_names", "pop_est", "applications_2025", "apps_1k", "hvi_popwtd", "pct_black_nh", "pct_hispanic", "hvi_tercile"] + \
       [f"{c['id']}_{suffix}" for c in comparisons for suffix in ("tercile", "color")]
master = master[keep + ["geometry"]]
write_geojson(DATA / "modzcta_bivariate.geojson", master, "modzcta", "modzcta",
              "Modified ZIP code areas with population-weighted HVI, 2025 cooling-assistance applications, 2020 Census shares, tercile classes and bivariate colors. hvi_popwtd is a derived population-weighted mean of an ordinal neighborhood index, not a published ZIP-level score.",
              MAP_SOURCES)
master.drop(columns="geometry").to_csv(DATA / "map_values.csv", index=False)
write_json(DATA / "legend.json", {
    "description": "Bivariate legend: palette[row][column], row = HVI tercile (0 low to 2 high), column = comparison tercile left to right. Edges are exact quantile cut points; bin 0 includes the minimum and the first cut, bin 1 is above the first cut up to the second, bin 2 is above the second cut. The application class is reversed after binning. Null stays null.",
    "sources": MAP_SOURCES, "palette": BIVARIATE_PALETTE, "no_data_color": NO_DATA, "background": "#EFEEED",
    "style": {"fillOpacity": 1, "color": "#ffffff", "weight": 0.5}, "hvi_edges": hvi_edges, "comparisons": comparisons,
})

city = census.loc[census.GeoType == "NYC2020"].iloc[0]
native = census.loc[census.GeoType == "NTA2020", ["GeoID", "Pop1"]].merge(hvi[["NTACode", "HVI_RANK"]], left_on="GeoID", right_on="NTACode", validate="one_to_one")
apps_valid = master.applications_2025.notna() & (master.pop_est > 0)
apps_total, apps_population = int(master.loc[apps_valid, "applications_2025"].sum()), int(master.loc[apps_valid, "pop_est"].sum())
assert int(city.Pop1) == NYC_POPULATION_2020 and len(native) == 197
assert apps_total == int(apps.applications_2025.sum()) and int(apps_valid.sum()) == 173
citywide = {
    "description": "Citywide benchmarks: 2020 Census population shares from the NYC total row; population-weighted mean of the published neighborhood HVI rank across scored neighborhoods; 2025 applications per 1,000 residents across mapped areas with application records. Aggregates of counts, not averages of area percentages.",
    "sources": MAP_SOURCES,
    "population_2020": int(city.Pop1),
    "black_non_hispanic_count": int(city.BNH), "pct_black_nh": float(city.BNH / city.Pop1 * 100),
    "hispanic_count": int(city.Hsp1), "pct_hispanic": float(city.Hsp1 / city.Pop1 * 100),
    "hvi_population_weighted_mean": float((native.Pop1 * native.HVI_RANK).sum() / native.Pop1.sum()),
    "hvi_scored_neighborhoods": len(native), "hvi_scored_population_2020": int(native.Pop1.sum()),
    "hvi_method": "Sum(neighborhood HVI rank × neighborhood 2020 population) / population of scored neighborhoods. A descriptive mean of an ordinal index.",
    "applications_2025": apps_total, "application_population_denominator": apps_population,
    "apps_per_1000": apps_total / apps_population * 1000,
    "application_areas_included": int(apps_valid.sum()), "application_areas_excluded": int((~apps_valid).sum()),
    "application_scope": "Mapped areas with a 2025 application record and positive pop_est: Σ applications / Σ pop_est × 1000. Excludes areas without records (including the zero-population pseudo-area) and applications from ZIP codes with no map area.",
    "tract_crosswalk": crosswalk_stats,
}
write_json(DATA / "citywide_context.json", citywide)
print(f"citywide: {citywide['pct_black_nh']:.1f}% Black non-Hispanic, {citywide['pct_hispanic']:.1f}% Hispanic; "
      f"population-weighted HVI {citywide['hvi_population_weighted_mean']:.2f}; {citywide['apps_per_1000']:.2f} applications per 1,000 residents")


wrote 04_heat_vulnerability_overlap/data/modzcta_bivariate.geojson (178 features, 8,536 vertices)
wrote 04_heat_vulnerability_overlap/data/legend.json
wrote 04_heat_vulnerability_overlap/data/citywide_context.json
citywide: 20.2% Black non-Hispanic, 28.3% Hispanic; population-weighted HVI 3.06; 3.16 applications per 1,000 residents


## Results

Cross-tabulations of the 178 areas by heat-vulnerability tercile (rows, low to high) and comparison tercile
(columns). For applications the columns run from *more* to *fewer* applications per 1,000, as in the legend.
The last table sums the interpolated tract counts within each HVI tercile's areas (the area-share pieces of step 1),
so the shares describe the areas' polygons under the uniform-density assumption. Both tables are written to `data/` so the
README figures can be traced.

These are co-locations, and none of this establishes causation or anything about program decisions. The share
of Black residents is one of the five inputs to the city's HVI (surface temperature, green space, air conditioning
access, median income, Black population), so the Black-population comparison is partly related by construction;
Hispanic share is not an input.

In [7]:
for c in comparisons:
    cols = ["more", "mid", "fewer"] if c["inverted"] else ["low", "mid", "high"]
    table = pd.crosstab(master.hvi_tercile, master[c["class_column"]]).rename(index={0: "low", 1: "mid", 2: "high"}, columns=dict(enumerate(cols)))
    table.index.name, table.columns.name = "HVI tercile", c["title"]
    display(table)

apps_by_tercile = master[apps_valid].groupby("hvi_tercile").agg(areas=("modzcta", "size"), applications_2025=("applications_2025", "sum"), pop_est=("pop_est", "sum"))
apps_by_tercile[["applications_2025", "pop_est"]] = apps_by_tercile[["applications_2025", "pop_est"]].astype(int)
apps_by_tercile["apps_per_1000"] = apps_by_tercile.applications_2025 / apps_by_tercile.pop_est * 1000
apps_by_tercile.rename(index={0: "low", 1: "mid", 2: "high"}).round(2).to_csv(DATA / "applications_by_hvi_tercile.csv")
print("Applications per 1,000 residents by HVI tercile (Σ applications / Σ pop_est, areas with records):", apps_by_tercile.apps_per_1000.round(2).to_dict())

tract_terciles = pieces.merge(master[["modzcta", "hvi_tercile"]], on="modzcta").dropna(subset=["hvi_tercile"])
composition = tract_terciles.groupby("hvi_tercile")[["Pop1", "BNH", "Hsp1"]].sum()
composition = pd.DataFrame({"population": composition.Pop1.round().astype(int), "black_non_hispanic": composition.BNH.round().astype(int),
                            "hispanic": composition.Hsp1.round().astype(int),
                            "pct_black_nh": (composition.BNH / composition.Pop1 * 100).round(1),
                            "pct_hispanic": (composition.Hsp1 / composition.Pop1 * 100).round(1)}).rename(index={0: "low", 1: "mid", 2: "high"})
composition.to_csv(DATA / "composition_by_hvi_tercile.csv")
composition


"Applications per 1,000 residents (2025)",more,mid,fewer
HVI tercile,,,
low,5,13,38
mid,15,26,17
high,38,18,3


"% Black, non-Hispanic (2020 Census)",low,mid,high
HVI tercile,,,
low,34,25,0
mid,23,28,8
high,2,7,50


"% Hispanic, any race (2020 Census)",low,mid,high
HVI tercile,,,
low,38,18,3
mid,9,25,25
high,13,15,31


Applications per 1,000 residents by HVI tercile (Σ applications / Σ pop_est, areas with records): {0.0: 1.46, 1.0: 2.65, 2.0: 4.61}


,population,black_non_hispanic,hispanic,pct_black_nh,pct_hispanic
hvi_tercile,,,,,
low,2119391,106438,338789,5.0,16.0
mid,3210304,288808,946756,9.0,29.5
high,3474322,1381584,1204754,39.8,34.7
